# Credit Card Fraud Detection Model

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc, f1_score
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data & EDA

In [2]:
df = pd.read_csv('portfolio_credit_card_fraud_synthetic_100500_rows.csv')
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nMissing values:")
print(df.isnull().sum().sum())

target_col = 'is_fraud'
print(f"\nTarget Column identified as: {target_col}")
print("Class Distribution:")
print(df[target_col].value_counts(normalize=True))

Dataset Shape: (100500, 24)

First 5 rows:
  transaction_id customer_id transaction_datetime  transaction_amount  \
0       T0000001     C013703      4/20/2025 20:16              694.88   
1       T0000002     C001527       11/6/2025 3:00              747.08   
2       T0000003     C017949      7/30/2025 21:17              488.38   
3       T0000004     C022905      5/30/2025 23:23             1619.48   
4       T0000005     C004981       8/7/2025 15:14             2129.08   

   transaction_hour  transaction_frequency_24h  avg_transaction_amount_30d  \
0                20                          3                     3721.47   
1                 3                          4                     2053.18   
2                21                          3                     2263.59   
3                23                          3                     1531.70   
4                15                          4                     1745.13   

   amount_deviation  distance_from_home_km  previ

## 2. Preprocess Data

In [3]:
X = df.drop(columns=[target_col])
y = df[target_col]

id_cols_to_drop = [col for col in ['transaction_id', 'customer_id', 'merchant_id', 'device_id'] if col in X.columns]
X = X.drop(columns=id_cols_to_drop)

import pandas.api.types as ptypes
for col in X.columns:
    if ptypes.is_numeric_dtype(X[col]):
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].fillna(X[col].mode()[0])

for col in X.select_dtypes(include=['object', 'string', 'category']).columns:
    num_unique = X[col].nunique()
    print(f"Column '{col}' has {num_unique} unique values.")
    if num_unique > 100:
        print(f"Dropping '{col}' due to high cardinality.")
        X = X.drop(columns=[col])

X = pd.get_dummies(X, drop_first=True)
print("Shape after one-hot encoding:", X.shape)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

Column 'transaction_datetime' has 91180 unique values.
Dropping 'transaction_datetime' due to high cardinality.
Column 'merchant_category' has 10 unique values.
Column 'merchant_city' has 10 unique values.
Column 'transaction_channel' has 4 unique values.
Column 'payment_method' has 2 unique values.
Column 'card_type' has 4 unique values.
Column 'device_type' has 5 unique values.
Shape after one-hot encoding: (100500, 43)

Training set size: 80400
Testing set size: 20100


## 3. Model Training

In [4]:
print("\n--- Training Logistic Regression ---")
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

print("\n--- Training Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)


--- Training Logistic Regression ---

--- Training Random Forest ---


,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

## 4. Evaluation & Save

In [9]:
def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"\n=== {name} Evaluation ===")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(recall, precision)
    print(f"PR-AUC: {pr_auc:.4f}")
    return pr_auc

lr_auc = evaluate_model(lr_model, X_test, y_test, "Logistic Regression")
rf_auc = evaluate_model(rf_model, X_test, y_test, "Random Forest")

best_model = rf_model if rf_auc >= lr_auc else lr_model
best_name = "Random Forest" if rf_auc >= lr_auc else "Logistic Regression"

print(f"\nBest model selected based on PR-AUC: {best_name}")
joblib.dump(best_model, 'best_fraud_model.pkl')
print("Model saved to 'best_fraud_model.joblib'")


=== Logistic Regression Evaluation ===
Confusion Matrix:
[[19591     0]
 [    1   508]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19591
           1       1.00      1.00      1.00       509

    accuracy                           1.00     20100
   macro avg       1.00      1.00      1.00     20100
weighted avg       1.00      1.00      1.00     20100

PR-AUC: 1.0000

=== Random Forest Evaluation ===
Confusion Matrix:
[[19591     0]
 [    0   509]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19591
           1       1.00      1.00      1.00       509

    accuracy                           1.00     20100
   macro avg       1.00      1.00      1.00     20100
weighted avg       1.00      1.00      1.00     20100

PR-AUC: 1.0000

Best model selected based on PR-AUC: Random Forest
Model saved to 'best_fraud_model.joblib'


In [8]:
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved to 'scaler.joblib'")

Scaler saved to 'scaler.joblib'


In [10]:
joblib.dump(X.columns.tolist(), "feature_names.joblib")
print("Feature names saved.")

Feature names saved.
